In [54]:
!pip install -q pandas pyarrow

In [55]:
from pathlib import Path
import pandas as pd

In [56]:
BASE_DIR = Path.cwd()
SHARED_DATA_DIR = (BASE_DIR / "../data/shared_data").resolve()

STAGING_DIR = SHARED_DATA_DIR / "staging"
MASTER_DIR = SHARED_DATA_DIR / "master"
LOGS_DIR = SHARED_DATA_DIR / "logs"

MASTER_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

VIDEOS_MASTER_FILE = MASTER_DIR / "videos_master.parquet"
COMMENTS_MASTER_FILE = MASTER_DIR / "comments_master.parquet"
RUNS_MASTER_FILE = LOGS_DIR / "runs_log_master.parquet"

In [57]:
def load_parquet_if_exists(path: Path) -> pd.DataFrame:
    if path.exists() and path.stat().st_size > 0:
        return pd.read_parquet(path)
    return pd.DataFrame()

def deduplicate_videos(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["video_published_at"] = pd.to_datetime(x["video_published_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["video_published_at", "fetched_at_utc"], ascending=[False, False], na_position="last")
    x = x.drop_duplicates(subset=["video_id"], keep="first")
    return x.reset_index(drop=True)

def deduplicate_comments(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    x = df.copy()
    x["published_at"] = pd.to_datetime(x["published_at"], utc=True, errors="coerce")
    x["updated_at"] = pd.to_datetime(x["updated_at"], utc=True, errors="coerce")
    x["fetched_at_utc"] = pd.to_datetime(x["fetched_at_utc"], utc=True, errors="coerce")
    x = x.sort_values(["updated_at", "published_at", "fetched_at_utc"], ascending=[False, False, False], na_position="last")
    x = x.drop_duplicates(subset=["comment_id"], keep="first")
    return x.reset_index(drop=True)

def merge_master_with_new(master: pd.DataFrame, new_df: pd.DataFrame, kind: str) -> pd.DataFrame:
    if master.empty and new_df.empty:
        return pd.DataFrame()

    if master.empty:
        combined = new_df.copy()
    elif new_df.empty:
        combined = master.copy()
    else:
        combined = pd.concat([master, new_df], ignore_index=True)

    if kind == "videos":
        return deduplicate_videos(combined)
    elif kind == "comments":
        return deduplicate_comments(combined)
    else:
        return combined.drop_duplicates().reset_index(drop=True)

In [58]:
video_files = sorted(STAGING_DIR.glob("*_videos_*.parquet"))
comment_files = sorted(STAGING_DIR.glob("*_comments_*.parquet"))
run_files = sorted(STAGING_DIR.glob("*_runs_*.parquet"))

print("Video staging files:", len(video_files))
print("Comment staging files:", len(comment_files))
print("Run log staging files:", len(run_files))

for f in video_files[:6]:
    print("VIDEO:", f.name)

for f in comment_files[:6]:
    print("COMMENT:", f.name)

for f in run_files[:6]:
    print("RUN:", f.name)

Video staging files: 6
Comment staging files: 6
Run log staging files: 6
VIDEO: leader_videos_20260402_212816.parquet
VIDEO: member_1_videos_20260402_210802.parquet
VIDEO: member_2_videos_20260402_214822.parquet
VIDEO: member_3_videos_20260402_215728.parquet
VIDEO: member_4_videos_20260402_220643.parquet
VIDEO: member_5_videos_20260402_221907.parquet
COMMENT: leader_comments_20260402_212816.parquet
COMMENT: member_1_comments_20260402_210802.parquet
COMMENT: member_2_comments_20260402_214822.parquet
COMMENT: member_3_comments_20260402_215728.parquet
COMMENT: member_4_comments_20260402_220643.parquet
COMMENT: member_5_comments_20260402_221907.parquet
RUN: leader_runs_20260402_212816.parquet
RUN: member_1_runs_20260402_210802.parquet
RUN: member_2_runs_20260402_214822.parquet
RUN: member_3_runs_20260402_215728.parquet
RUN: member_4_runs_20260402_220643.parquet
RUN: member_5_runs_20260402_221907.parquet


In [59]:
video_dfs = [pd.read_parquet(f) for f in video_files] if video_files else []
comment_dfs = [pd.read_parquet(f) for f in comment_files] if comment_files else []
run_dfs = [pd.read_parquet(f) for f in run_files] if run_files else []

staging_videos = pd.concat(video_dfs, ignore_index=True) if video_dfs else pd.DataFrame()
staging_comments = pd.concat(comment_dfs, ignore_index=True) if comment_dfs else pd.DataFrame()
staging_runs = pd.concat(run_dfs, ignore_index=True) if run_dfs else pd.DataFrame()

print("Staging videos rows:", 0 if staging_videos.empty else len(staging_videos))
print("Staging comments rows:", 0 if staging_comments.empty else len(staging_comments))
print("Staging runs rows:", 0 if staging_runs.empty else len(staging_runs))

Staging videos rows: 7482
Staging comments rows: 86602
Staging runs rows: 427


In [60]:
videos_master_old = load_parquet_if_exists(VIDEOS_MASTER_FILE)
comments_master_old = load_parquet_if_exists(COMMENTS_MASTER_FILE)
runs_master_old = load_parquet_if_exists(RUNS_MASTER_FILE)

print("Existing master videos:", 0 if videos_master_old.empty else len(videos_master_old))
print("Existing master comments:", 0 if comments_master_old.empty else len(comments_master_old))
print("Existing master runs:", 0 if runs_master_old.empty else len(runs_master_old))

Existing master videos: 7082
Existing master comments: 78413
Existing master runs: 427


In [61]:
videos_master_new = merge_master_with_new(videos_master_old, staging_videos, kind="videos")
comments_master_new = merge_master_with_new(comments_master_old, staging_comments, kind="comments")

if runs_master_old.empty and staging_runs.empty:
    runs_master_new = pd.DataFrame()
elif runs_master_old.empty:
    runs_master_new = staging_runs.drop_duplicates().reset_index(drop=True)
elif staging_runs.empty:
    runs_master_new = runs_master_old.drop_duplicates().reset_index(drop=True)
else:
    runs_master_new = pd.concat([runs_master_old, staging_runs], ignore_index=True).drop_duplicates().reset_index(drop=True)

print("Merged master videos:", 0 if videos_master_new.empty else len(videos_master_new))
print("Merged master comments:", 0 if comments_master_new.empty else len(comments_master_new))
print("Merged master runs:", 0 if runs_master_new.empty else len(runs_master_new))

Merged master videos: 7082
Merged master comments: 78413
Merged master runs: 427


In [62]:
if not videos_master_new.empty:
    print("Unique video_id:", videos_master_new["video_id"].nunique(), " / rows:", len(videos_master_new))

if not comments_master_new.empty:
    print("Unique comment_id:", comments_master_new["comment_id"].nunique(), " / rows:", len(comments_master_new))

Unique video_id: 7082  / rows: 7082
Unique comment_id: 78413  / rows: 78413


In [63]:
display(videos_master_new.head())
display(comments_master_new.head())
display(runs_master_new.head())

,video_id,channel_id,channel_title,title,description,tags,category_id,default_language,default_audio_language,video_published_at,duration,view_count,like_count,comment_count,search_keyword,fetched_at_utc,fetched_by
0,dkjMvngrY3A,UCGdK3EmgcNSuTrv3PW_j5Rw,KPLikedIt,NEW Toddler Travel Essential That Fits in Your...,#gifted This toddler travel setup is GENIUS 🤯\...,,22,en-US,en-US,2026-04-02 14:22:54+00:00,PT37S,0,1.0,0.0,carry on essentials travel setup,2026-04-02 14:26:06+00:00,member_5
1,C3tIQXanODo,UCBEP0vydSQUqp4AxI6idtAg,Green Storage Canada,Green Storage Canada Customer Review - Scarbor...,Welcome to Green Storage! Hear from one of our...,self storage | storage unit rental | rent a st...,22,en,en,2026-04-02 14:02:20+00:00,PT33S,0,0.0,0.0,safe storage for watches,2026-04-02 14:07:34+00:00,member_4
2,_H1xpoWdyr4,UCv95QBbZEBlgp4cLz_pdTdQ,PomInOz Reviews,The Ultimate Travel Companion? Snap PowerPack ...,If you’re tired of carrying a tangled mess of ...,,28,en,en-AU,2026-04-02 14:00:00+00:00,PT45M28S,0,0.0,0.0,power bank travel setup,2026-04-02 14:02:19+00:00,member_3
3,C4bLDNWR3IQ,UCdL9M8nNerv7xQdU9l1evXw,pringles edits,scientists can now store memories on a hard dr...,,,22,en,en,2026-04-02 13:52:10+00:00,PT50S,2,1.0,1.0,how to store hard drive,2026-04-02 13:59:00+00:00,member_3
4,7whd8EtfvIU,UC1ZFR_9wxTlSNJ3Q_50RwCA,FT Digital Computer,Ultimate Budget Gaming PC 💥 Ryzen 5 + RX 580 B...,#gamingpc #budgetgamingpc #pcbuild #ryzen5 ...,,28,en,ta,2026-04-02 13:49:58+00:00,PT1M5S,2,0.0,0.0,ssd accessories setup,2026-04-02 14:03:25+00:00,member_3


,comment_id,video_id,thread_id,parent_comment_id,is_reply,author_channel_id,author_display_name,text_original,text_display,like_count,published_at,updated_at,total_reply_count,search_keyword,fetched_at_utc,fetched_by
0,Ugwho3G5j5CmtjU6V-V4AaABAg,iDi5njMvGuA,Ugwho3G5j5CmtjU6V-V4AaABAg,None,False,UCOvkp7JtSKKS7ohbsThAGlA,@Sweat-1234,Toy,Toy,0,2026-04-02 14:18:12+00:00,2026-04-02 14:18:12+00:00,0.0,tools daily carry,2026-04-02 14:24:17+00:00,member_5
1,UgwLBwgFppldIJJXh1B4AaABAg.AV4rxmTeMzlAV5ynrvbKnP,KI3l42XrTjE,UgwLBwgFppldIJJXh1B4AaABAg,UgwLBwgFppldIJJXh1B4AaABAg,True,UC_rI3y1DzDULTr-UIvshiwg,@PackHacker,Thanks for the suggestion! I’ll pass it along.,Thanks for the suggestion! I’ll pass it along.,0,2026-04-02 14:16:34+00:00,2026-04-02 14:16:34+00:00,NaN,carry on essentials daily carry,2026-04-02 14:26:29+00:00,member_5
2,UgzLcn-721d0bczpSM14AaABAg,URKIJkEgm0o,UgzLcn-721d0bczpSM14AaABAg,None,False,UCpfWT61xOMayqmK_HOKoVhQ,@domenicherrrera3630,He bought the shit. Quit complaining,He bought the shit. Quit complaining,0,2026-04-02 14:08:17+00:00,2026-04-02 14:08:17+00:00,0.0,how to store cards,2026-04-02 14:10:16+00:00,member_4
3,UgyShCJWt12baH2GcZl4AaABAg,4o1OtakSVdA,UgyShCJWt12baH2GcZl4AaABAg,None,False,UCsMA_hNQJpR52ruvdjmZ3cQ,@djdavis1420,Got my Deck in Summer 2022. Switched my old In...,Got my Deck in Summer 2022. Switched my old In...,0,2026-04-02 14:07:10+00:00,2026-04-02 14:07:35+00:00,0.0,how to store steam deck,2026-04-02 14:07:51+00:00,member_4
4,UgyqA3hzq9xLKURijSx4AaABAg,A0IaFtazG8E,UgyqA3hzq9xLKURijSx4AaABAg,None,False,UCh45s0buES6iCOmJMBdrXRw,@noormajeed7144,Kiya white powder lagana ha?,Kiya white powder lagana ha?,0,2026-04-02 14:06:01+00:00,2026-04-02 14:06:01+00:00,0.0,how to protect makeup,2026-04-02 14:08:40+00:00,member_4


,run_tag,team_member,search_keyword,published_after,candidate_video_ids,videos_fetched,comments_fetched,run_started_at_utc,logged_at_utc
0,20260402_212816,leader,desk setup tech gadgets,2026-03-03T13:28:16Z,100,100,405,2026-04-02T13:28:16Z,2026-04-02T13:29:00Z
1,20260402_212816,leader,gaming setup accessories,2026-03-03T13:28:16Z,100,100,883,2026-04-02T13:28:16Z,2026-04-02T13:29:35Z
2,20260402_212816,leader,what's in my bag tech,2026-03-03T13:28:16Z,83,83,1428,2026-04-02T13:28:16Z,2026-04-02T13:30:13Z
3,20260402_212816,leader,college tech essentials,2026-03-03T13:28:16Z,2,2,5,2026-04-02T13:28:16Z,2026-04-02T13:30:16Z
4,20260402_212816,leader,camera bag setup,2026-03-03T13:28:16Z,68,68,798,2026-04-02T13:28:16Z,2026-04-02T13:30:40Z


In [64]:
# 保存 master 数据
videos_master_new.to_parquet(VIDEOS_MASTER_FILE, index=False)
comments_master_new.to_parquet(COMMENTS_MASTER_FILE, index=False)
runs_master_new.to_parquet(RUNS_MASTER_FILE, index=False)

print("✅ Master data saved!")

✅ Master data saved!


In [ ]:
# 保存为csv
videos_master_new.to_csv(MASTER_DIR / "videos_master.csv", index=False)
comments_master_new.to_csv(MASTER_DIR / "comments_master.csv", index=False)
runs_master_new.to_csv(MASTER_DIR / "runs_master.csv", index=False)

print("✅ 成功保存csv!")
